In [1]:
import json
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

print(f"Project root: {ROOT_DIR}")

Project root: c:\Users\MyBook Hype AMD\Documents\Projects\Chessys


In [2]:
DATA_PATH = ROOT_DIR / "data" / "processed" / "dataset_poc.parquet"
df = pd.read_parquet(DATA_PATH)

FEATURE_COLS = [
    "material_balance",
    "tactical_tension",
    "hanging_pieces",
    "king_openness",
    "legal_moves_count",
    "mobility_ratio",
    "is_check",
    "absolute_pins",
    "has_queen_turn",
    "has_queen_opp",
    "player_elo",
    "ply",
]
TARGET_COL = "is_blunder"

missing = [c for c in FEATURE_COLS + [TARGET_COL] if c not in df.columns]
assert not missing, f"Kolom wajib tidak ditemukan: {missing}"

print(f"Total baris data: {len(df):,}")
print(f"Fitur prediktor terdaftar ({len(FEATURE_COLS)}): {FEATURE_COLS}")

Total baris data: 10,000
Fitur prediktor terdaftar (12): ['material_balance', 'tactical_tension', 'hanging_pieces', 'king_openness', 'legal_moves_count', 'mobility_ratio', 'is_check', 'absolute_pins', 'has_queen_turn', 'has_queen_opp', 'player_elo', 'ply']


In [3]:
bool_cols = ["is_check", "has_queen_turn", "has_queen_opp", "is_blunder"]
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(int)

X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].copy()

X.replace([np.inf, -np.inf], np.nan, inplace=True)
if X.isna().sum().sum() > 0:
    print("Mengisi missing/inf values dengan median...")
    X.fillna(X.median(), inplace=True)

assert X.select_dtypes(include=[np.number]).shape[1] == len(
    FEATURE_COLS
), "Semua fitur wajib berupa numerik."
print("Sanitasi data berhasil: 0 NaN / Inf terdeteksi.")

✓ Sanitasi data berhasil: 0 NaN / Inf terdeteksi.


In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

def print_split_stats(name, y_subset):
    blunder_rate = y_subset.mean() * 100
    print(f"  - {name:<12}: {len(y_subset):>6,} baris | Blunder: {blunder_rate:.2f}%")

print("Distribusi Partisi Data:")
print_split_stats("Train Set", y_train)
print_split_stats("Val Set", y_val)
print_split_stats("Test Set", y_test)

Distribusi Partisi Data:
  - Train Set   :  7,000 baris | Blunder: 9.93%
  - Val Set     :  1,500 baris | Blunder: 9.93%
  - Test Set    :  1,500 baris | Blunder: 9.93%


In [5]:
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
ARTIFACTS_DIR = ROOT_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.concat([X_train, y_train], axis=1)
val_df = pd.concat([X_val, y_val], axis=1)
test_df = pd.concat([X_test, y_test], axis=1)

train_df.to_parquet(PROCESSED_DIR / "train.parquet", index=False)
val_df.to_parquet(PROCESSED_DIR / "val.parquet", index=False)
test_df.to_parquet(PROCESSED_DIR / "test.parquet", index=False)

with open(ARTIFACTS_DIR / "feature_names.json", "w") as f:
    json.dump(FEATURE_COLS, f, indent=2)

print("Berkas partisi berhasil disimpan di data/processed/:")
print("  - train.parquet")
print("  - val.parquet")
print("  - test.parquet")
print("Kontrak fitur berhasil disimpan di artifacts/feature_names.json")

✓ Berkas partisi berhasil disimpan di data/processed/:
  - train.parquet
  - val.parquet
  - test.parquet
✓ Kontrak fitur berhasil disimpan di artifacts/feature_names.json
